In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])  

In [10]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [11]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [12]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [13]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [14]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [15]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [16]:
df['text'] = df['text'].apply(remove_punc)

In [17]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [18]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)
     

In [23]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [24]:
import nltk
print(nltk.__version__)

3.9.2


In [25]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [41]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yadav\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yadav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [42]:
stop_words = set(stopwords.words('english'))

In [43]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [44]:
def remove(txt):
  words =  word_tokenize(txt)
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [45]:
df['text'] = df['text'].apply(remove)

In [46]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [47]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [48]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['emotion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [49]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

In [50]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_bow, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [51]:
y_pred = model.predict(X_test_bow)

In [52]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.7678125


In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [54]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [55]:
y_pred = model.predict(X_test_tfidf)

In [56]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.6621875
              precision    recall  f1-score   support

           0       0.68      0.92      0.78       933
           1       0.94      0.28      0.43       432
           2       1.00      0.04      0.08       261
           3       1.00      0.01      0.02       115
           4       0.98      0.22      0.37       387
           5       0.61      0.97      0.75      1072

    accuracy                           0.66      3200
   macro avg       0.87      0.41      0.40      3200
weighted avg       0.76      0.66      0.59      3200



In [60]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X = df['text']
y = df['emotion']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Prediction
y_pred = model.predict(X_test_tfidf)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8509375
              precision    recall  f1-score   support

           0       0.89      0.93      0.91       933
           1       0.89      0.76      0.82       432
           2       0.90      0.57      0.70       261
           3       0.86      0.47      0.61       115
           4       0.88      0.76      0.81       387
           5       0.80      0.96      0.87      1072

    accuracy                           0.85      3200
   macro avg       0.87      0.74      0.79      3200
weighted avg       0.86      0.85      0.85      3200



In [61]:
import os
import joblib

# model folder create karo
os.makedirs("model", exist_ok=True)

# Logistic Regression model save karo
joblib.dump(model, "model/model.pkl")

# TF-IDF vectorizer save karo
joblib.dump(tfidf, "model/tfidf.pkl")

print("Files saved successfully!")
print(os.listdir("model"))

Files saved successfully!
['model.pkl', 'tfidf.pkl']


In [62]:
import os

print(os.getcwd())
print(os.path.exists("model/model.pkl"))
print(os.path.exists("model/tfidf.pkl"))

C:\Users\yadav
True
True


In [63]:
from IPython.display import FileLink, display

display(FileLink("model/model.pkl"))
display(FileLink("model/tfidf.pkl"))

C:\Users\yadav\model\model.pkl

C:\Users\yadav\model\tfidf.pkl

In [64]:
joblib.dump(model, "model/model.pkl")
joblib.dump(tfidf, "model/tfidf.pkl")

['model/tfidf.pkl']